# Python 03 — Functions and Scope

**Roadmap position:** Python → Core → Functions and scope.

Functions turn a piece of logic into a reusable contract: named inputs, one responsibility, a predictable return value, and clear failures. Scope determines which names are available at each point in the program.

## Outcome

You will be able to write small functions, choose parameters and return values, validate input, explain local versus global scope, and debug a common scope error.

## 1. A function is a contract

A strong function has a descriptive name, focused responsibility, explicit input and output types, and a documented reason to raise an error. Its caller should not need to inspect its implementation to use it correctly.

Type hints are documentation and static-analysis support; they do not automatically validate runtime values.

In [1]:
def calculate_average_study_hours(total_hours: float, recorded_days: int) -> float:
    """Return average study hours per day for a valid, non-empty period."""
    if total_hours < 0:
        raise ValueError('total_hours must not be negative')
    if recorded_days <= 0:
        raise ValueError('recorded_days must be greater than zero')
    return total_hours / recorded_days

weekly_average = calculate_average_study_hours(14.5, 7)
print(f'Average: {weekly_average:.2f} hours/day')

Average: 2.07 hours/day


## Your turn 1 — define a focused function

Implement `calculate_completion_rate(completed: int, total: int) -> float`. It returns the percentage completed, from `0.0` to `100.0`.

Contract:

- reject a negative `completed` or `total`;
- reject `total == 0`;
- reject `completed > total`;
- return a `float`; and
- use clear `ValueError` messages.

Do not print inside the function. The function should return data; its caller decides how to display it.

In [2]:
# YOUR TURN
def calculate_completion_rate(completed: int, total: int) -> float:
    if completed <0 or total < 0:
        raise ValueError
    if total == 0:
        raise ValueError
    if completed > total:
        raise ValueError
    return completed/total*100


In [3]:
# Checks — run after implementing calculate_completion_rate.
assert calculate_completion_rate(0, 10) == 0.0
assert calculate_completion_rate(1, 4) == 25.0
assert calculate_completion_rate(8, 8) == 100.0

for invalid_arguments in ((-1, 10), (1, -10), (0, 0), (11, 10)):
    try:
        calculate_completion_rate(*invalid_arguments)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{invalid_arguments} should raise ValueError')

print('Completion-rate checks passed.')

Completion-rate checks passed.


## 2. Parameters, arguments, and defaults

Parameters are names in a function definition. Arguments are the actual values supplied when calling it. A default makes an argument optional, but defaults should be stable values such as strings, numbers, or `None`—not mutable containers such as `[]` or `{}`.

Keyword arguments make calls clearer when several arguments have the same type.

In [ ]:
def build_progress_message(
    topic_name: str,
    completed: int,
    total: int,
    prefix: str = 'Progress',
) -> str:
    """Build a display message; formatting remains separate from calculation."""
    percentage = calculate_average_study_hours(float(completed * 100), total)
    return f'{prefix}: {topic_name} is {percentage:.1f}% complete'

print(build_progress_message('Python', 3, 10))
print(build_progress_message(topic_name='SQL', completed=4, total=5, prefix='Status'))

## Your turn 2 — separate calculation from presentation

Implement `build_application_message(company_name: str, application_count: int, label: str = 'Applications') -> str`.

Return a message in this shape: `Applications: Google — 3 submitted`. Reject a negative count. Keep this function side-effect free: return a string, do not print it.

Call it once with positional arguments and once with keyword arguments.

In [14]:
# YOUR TURN
def build_application_message(
    company_name: str,
    application_count: int,
    label: str = 'Applications',
) -> str:
     if application_count < 0:
        raise ValueError("negative value")

     return f"{label}: {company_name} — {application_count} submitted"


## 3. Scope: where a name exists

A name created inside a function is local to that function. A name created outside is global to the notebook/module. A function can read a global name, but assigning to a name inside a function makes it local unless explicitly declared otherwise.

Avoid mutable global state in production code. Prefer passing state in and returning new state out: it is easier to test, reuse, and reason about.

In [15]:
default_timeout_seconds = 10  # Global/module scope.


def request_timeout_message() -> str:
    # Reading a global name is allowed.
    return f'Request timeout: {default_timeout_seconds} seconds'


print(request_timeout_message())
# `default_timeout_seconds` is still available here.
print(default_timeout_seconds)

Request timeout: 10 seconds
10


## Debugging drill — local assignment versus global state

The following function fails with `UnboundLocalError`. Run it, read the traceback, and explain why Python treats `request_count` as local.

Then refactor it without `global`: make the count an input and return the next count. Update the call site accordingly. This is the production-friendly design.

**Interview question:** Why is avoiding mutable global state useful for tests and concurrent programs?

In [17]:
# DEBUG ME
request_count = 0


def record_request(request_count) -> int:
    request_count += 1
    return request_count


print(record_request(request_count))

1


## Your turn 3 — compose small functions

Write two functions:

1. `is_valid_email(email: str) -> bool`: return `True` only when the stripped email contains exactly one `@`, has at least one character before it, and has at least one character after it. This is intentionally a simple rule, not full email validation.
2. `registration_status(email: str) -> str`: use `is_valid_email`; return `'accepted'` for a valid email and `'rejected'` otherwise.

The second function must call the first one. This is composition: small functions with one job working together.

In [18]:
def is_valid_email(email: str) -> bool:
    cleaned_email = email.strip()

    if cleaned_email.count("@") != 1:
        return False

    local_part, domain_part = cleaned_email.split("@")
    return bool(local_part) and bool(domain_part)


def registration_status(email: str) -> str:
    if is_valid_email(email):
        return "accepted"
    return "rejected"

In [19]:
# Checks — run after implementing both functions.
assert is_valid_email('sam@example.com') is True
assert is_valid_email(' sam@example.com ') is True
assert is_valid_email('sam@@example.com') is False
assert is_valid_email('@example.com') is False
assert is_valid_email('sam@') is False
assert registration_status('sam@example.com') == 'accepted'
assert registration_status('not-an-email') == 'rejected'
print('Function-composition checks passed.')

Function-composition checks passed.


## Exit interview check

Answer without running code:

1. What is the difference between a parameter and an argument?
2. Why should a function return data instead of printing it?
3. When should a function raise `ValueError`?
4. What is local scope? Why did the debugging exercise fail?
5. Why is passing state in and returning state out safer than changing a global variable?

When finished, send me the three implementations, the fixed debugging cell, and any failed output. I’ll review your work without editing it. Then we will proceed to **lists, tuples, sets, and dictionaries**.